In [7]:
from datasets import load_dataset
import json

# Load SST-2 (classification)
sst2 = load_dataset("glue", "sst2", split="train")

# Load STS-B (regression)
stsb = load_dataset("glue", "stsb", split="train")

combined_data = []

# Process SST-2 (classification)
for sample in sst2:
    combined_data.append({
        "sentence": sample["sentence"],
        "classification_label": sample["label"],  # 0 or 1
        "task_id": 0  # 0 = classification
    })

# Process STS-B (regression)
for sample in stsb:
    # Combine the two sentences into one input (you can join with a separator)
    sentence_input = sample["sentence1"] + " </s> " + sample["sentence2"]
    combined_data.append({
        "sentence": sentence_input,
        "regression_value": float(sample["label"]),  # similarity score
        "task_id": 1  # 1 = regression
    })

# Save as one JSON array
with open("glue_multitask_train.json", "w", encoding="utf-8") as f:
    json.dump(combined_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(combined_data)} samples to glue_multitask_train.json")


Saved 73098 samples to glue_multitask_train.json


In [ ]:
from datasets import load_dataset
import json

# Load datasets
sst2 = load_dataset("glue", "sst2", split="train")
stsb = load_dataset("glue", "stsb", split="train")

combined_data = []

label_map = {0: "negative", 1: "positive"}

# SST-2 (classification)
for sample in sst2:
    combined_data.append({
        "instruction": "Please answer the following question with positive or negative. "
                       f"The sentence is: {sample['sentence']}",
        "input": "",
        "output": label_map[sample["label"]],
        "task_id": 0
    })

# STS-B (regression as text, 0–5 scale)
for sample in stsb:
    sentence_input = sample["sentence1"] + " </s> " + sample["sentence2"]
    combined_data.append({
        "instruction": "Rate the semantic similarity between the following two sentences on a scale from 0 to 5. "
                       f"{sentence_input}",
        "input": "",
        "output": str(sample["label"]),  # keep 0–5, convert to string
        "task_id": 1
    })

# Save
with open("glue_multitask_train.json", "w", encoding="utf-8") as f:
    json.dump(combined_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(combined_data)} training samples")


Saved 2372 validation samples


In [3]:
from datasets import load_dataset
import json
import random
# Load datasets
sst2 = load_dataset("glue", "sst2", split="train")
stsb = load_dataset("glue", "stsb", split="train")

combined_data = []

label_map = {0: "negative", 1: "positive"}

# SST-2 (classification)
for sample in sst2:
    combined_data.append({
        "instruction": "Please answer the following question with positive or negative. "
                       f"The sentence is: {sample['sentence']}",
        "input": "",
        "output": label_map[sample["label"]],
        "task_id": 0
    })

# STS-B (regression as text, 0–5 scale)
for sample in stsb:
    sentence_input = sample["sentence1"] + " </s> " + sample["sentence2"]
    combined_data.append({
        "instruction": "Rate the semantic similarity between the following two sentences on a scale from 0 to 5. "
                       f"{sentence_input}",
        "input": "",
        "output": str(sample["label"]),  # keep 0–5, convert to string
        "task_id": 1
    })


random.shuffle(combined_data)
# Save
with open("glue_multitask_code_train.json", "w", encoding="utf-8") as f:
    json.dump(combined_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(combined_data)} training samples")


Saved 73098 training samples


In [219]:
from datasets import load_dataset
import json
import random
# Load datasets
rte = load_dataset("glue", "rte", split="train")

rte_data = []


label_map = {0: "not_entailment", 1: "entailment"}

# SST-2 (classification)
for sample in rte:
    rte_data.append({
        "instruction":    f"Does the first sentence entail the second?\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}\n\nAnswer format: entailment/not_entailment",
        "input": "",
        "output": label_map[sample["label"]],
        "task_id": 6
    })




# Save
with open("rte.json", "w", encoding="utf-8") as f:
    json.dump(rte_data, f, ensure_ascii=False, indent=2)

    

print(f"Saved {len(rte_data)} training samples")



Saved 2490 training samples


In [15]:
temp = load_dataset("glue", "ax", split="train")

    if task == "rte":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}\n\nAnswer format: entailment/not_entailment"


ValueError: Unknown split "train". Should be one of ['test'].

In [19]:
import json
import random

In [105]:
##### glue and superglue

GLUE_SUPERGLUE_TASKS = [
    
    # GLUE
   # {"name": "ax", "group": "glue"}, this has test split
    #{"name": "mnli", "group": "glue"},
    #{"name": "cola", "group": "glue"},
    #{"name": "sst2", "group": "glue"},
    #{"name": "mrpc", "group": "glue"},
    #{"name": "qqp", "group": "glue"},
    #{"name": "qnli", "group": "glue"},
    #{"name": "rte", "group": "glue"},
    #{"name": "wnli", "group": "glue"}, # not in my paper
    #{"name": "stsb", "group": "glue"},

    # SuperGLUE
    {"name": "multirc", "group": "superglue"},
    {"name": "wsc", "group": "superglue"},
    {"name": "boolq", "group": "superglue"},
    {"name": "cb", "group": "superglue"},
    {"name": "wic", "group": "superglue"},
]


TASK_ID = {task["name"]: i for i, task in enumerate(GLUE_SUPERGLUE_TASKS)}

TASK_ID


{'multirc': 0, 'wsc': 1, 'boolq': 2, 'cb': 3, 'wic': 4}

In [107]:
from datasets import ClassLabel

def get_label_map(dataset):
    label_feat = dataset.features["label"]
    if isinstance(label_feat, ClassLabel):
        return {i: name for i, name in enumerate(label_feat.names)}
    return None  # regression


In [119]:
def build_instruction(task, sample):
    '''
    if task == "sst2":
        return f"Classify the sentiment of the sentence as positive or negative.\nSentence: {sample['sentence']}\n\nAnswer format: postive/negative"
    if task == "ax":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['premise']}\nSentence 2: {sample['hypothesis']}\n\nAnswer format: entailment/neutral/contradiction"
    if task == "mnli":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['premise']}\nSentence 2: {sample['hypothesis']}\n\nAnswer format: entailment/neutral/contradiction"
        
    if task == "cola":
        return f"Is the following sentence grammatically acceptable?\nSentence: {sample['sentence']}\n\nAnswer format: unacceptable/acceptable "

    if task == "mrpc":
        return f"Do the following two sentences have the same meaning?\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}\n\nAnswer format: not_equivalent/equivalent"

    if task == "qqp":
        return f"Are the following two questions duplicates?\nQuestion 1: {sample['question1']}\nQuestion 2: {sample['question2']}\n\nAnswer format: not_duplicate/duplicate"

    if task == "qnli":
        return f"Does the sentence answer the question?\nQuestion: {sample['question']}\nSentence: {sample['sentence']}\n\nAnswer format: entailment/not_entailment"

    if task == "rte":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}\n\nAnswer format: entailment/not_entailment"

    if task == "wnli":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}\n\nAnswer format: entailment/not_entailment"

    if task == "stsb":
        return f"Rate the semantic similarity between the two sentences on a scale from 0 to 5.\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}"
'''
    # SuperGLUE
    '''if task == "axb":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}\n\nAnswer format: entailment/not_entailment"
        
    if task == "axg":
        return f"Determine the relationship between the premise and hypothesis.\nPremise: {sample['premise']}\nHypothesis: {sample['hypothesis']}\n\nAnswer format: entailment/not_entailment"
'''
    if task == "boolq":
        return f"Answer the question based on the passage.\nPassage: {sample['passage']}\nQuestion: {sample['question']}\n\nAnswer format: False/True"
    if task=="multirc":
            return f"""Determine whether the candidate answer correctly answers the question based on the paragraph.

                    Paragraph:
                    {sample['paragraph']}
                    
                    Question: {sample['question']}
                    
                    Candidate Answer:
                    {sample['answer']}
                    
                    Answer Format: True / False
                    """
    if task == "wsc":
            return f"""Coreference Resolution Task: 

                        Sentence:
                        "{sample['text']}"
                      
                        Candidates:
                        A. {sample['span1_text']}
                        B. {sample['span2_text']}
                        Question: Does the pronoun refer to candidate A?
                         Answer: True or False. \n\nAnswer format: False/True"""

    if task =="wic":
        return f"""Decide whether the highlighted word has the same meaning in two different sentences.

                        Word: {sample['word']}
                        
                        Sentence 1: {sample['sentence1']}
                        Sentence 2: {sample['sentence2']}
                        
                        Answer Format: True/False
                        """  

    if task == "cb":
        return f"Determine the relationship between the premise and hypothesis.\nPremise: {sample['premise']}\nHypothesis: {sample['hypothesis']}\n\nAnswer format: entailment/neutral/contradiction"

    '''if task == "copa":
        direction = sample["question"]  
        return (
            f"Determine which option is the more plausible {direction} of the premise.\n"
            f"Premise: {sample['premise']}\n"
            f"Choice 1: {sample['choice1']}\n"
            f"Choice 2: {sample['choice2']}\n\n"
            f"Answer format: choice1/choice2"
        ) '''

    raise ValueError(f"Unknown task {task}")


In [129]:
from datasets import load_dataset

def build_mtl_dataset(task_defs, split="train"):
    all_data = []

    for task_def in task_defs:
        name = task_def["name"]
        group = task_def["group"]
        task_id = TASK_ID[name]

        if group == "glue":
            print(name)
            dataset = load_dataset("glue", name, split=split)
        else:
            print(name)
            dataset = load_dataset("super_glue", name, split=split)

        label_map = get_label_map(dataset)

        for sample in dataset:
            instruction = build_instruction(name, sample)

            if label_map is None:
                output = str(float(sample["label"]))  # STS-B
            else:
                output = label_map[sample["label"]]

            all_data.append({
                "instruction": instruction,
                "input": "",
                "output": output,
                #"answer": output,
                "task_id": task_id
            })

    return all_data


In [131]:
train_data = build_mtl_dataset(GLUE_SUPERGLUE_TASKS, split="train")

print(len(train_data))
print(train_data[0])


multirc
wsc
boolq
cb
wic
42902
{'instruction': 'Determine whether the candidate answer correctly answers the question based on the paragraph.\n\n                    Paragraph:\n                    While this process moved along, diplomacy continued its rounds. Direct pressure on the Taliban had proved unsuccessful. As one NSC staff note put it, "Under the Taliban, Afghanistan is not so much a state sponsor of terrorism as it is a state sponsored by terrorists." In early 2000, the United States began a high-level effort to persuade Pakistan to use its influence over the Taliban. In January 2000, Assistant Secretary of State Karl Inderfurth and the State Department\'s counterterrorism coordinator, Michael Sheehan, met with General Musharraf in Islamabad, dangling before him the possibility of a presidential visit in March as a reward for Pakistani cooperation. Such a visit was coveted by Musharraf, partly as a sign of his government\'s legitimacy. He told the two envoys that he would mee

In [133]:
random.shuffle(train_data)
# Save
with open("super_glue_code_train_dataset.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

In [135]:
from datasets import load_dataset

data_path = r"C:\Users\thara\super_glue_code_train_dataset.json"

data_test = load_dataset("json", data_files=data_path)
data_test

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'task_id'],
        num_rows: 42902
    })
})

In [259]:
#####  mmlu

MMLU_TASKS = [
    # MMLU
   # {"name": "ax", "group": "glue"}, this has test split
    {"name": "econometrics"},
    {"name": "high_school_macroeconomics"},
    {"name": "high_school_microeconomics"},
    {"name": "business_ethics" },
    {"name": "management"},
    {"name": "marketing"},
    {"name": "abstract_algebra"},
    {"name": "college_mathematics"}, 
    {"name": "elementary_mathematics"},
    {"name": "high_school_mathematics"}, 
    {"name": "high_school_statistics"},

    # BigBench

   # {"name": "multiemo_zero_shot", "group": "BigBench"},

    
]

#groups = set( s["group"] for s in MMLU_TASKS)

#group_to_taskid = {group: idx for idx, group in enumerate(groups)}
TASK_ID = {task["name"]: i for i, task in enumerate(MMLU_TASKS)}

print (TASK_ID)




{'econometrics': 0, 'high_school_macroeconomics': 1, 'high_school_microeconomics': 2, 'business_ethics': 3, 'management': 4, 'marketing': 5, 'abstract_algebra': 6, 'college_mathematics': 7, 'elementary_mathematics': 8, 'high_school_mathematics': 9, 'high_school_statistics': 10}


In [261]:
def build_instruction(task, sample):

    prompt=f"""The following is a multiple choice question. Understand the question and  select only one choice . 

            Question: {sample['question']}\n Choices:
            A. {sample['choices'][0]}
            B. {sample['choices'][1]}
            C. {sample['choices'][2]}
            D. {sample['choices'][3]}

            Answer Format: A/B/C/D
            """
        
    return prompt


    raise ValueError(f"Unknown task {task}")


In [263]:
from datasets import ClassLabel

def get_label_map(dataset):
    label_feat = dataset.features["answer"]
    if isinstance(label_feat, ClassLabel):
        return {i: name for i, name in enumerate(label_feat.names)}
    return None  # regression

In [269]:
from datasets import load_dataset

def build_mtl_dataset(task_defs, split="train"):
    all_data = []

    for task_def in task_defs:
        name = task_def["name"]
        #group = task_def["group"]
        task_id = TASK_ID[name]

        #if group == "mmlu":
        print(name)
        dataset = load_dataset("cais/mmlu", name, split=split)
        

        label_map = get_label_map(dataset)

        for sample in dataset:
            instruction = build_instruction(name, sample)


            output = label_map[sample["answer"]]

            all_data.append({
                "instruction": instruction,
                "input": "",
                "output": output,
                "task_id": task_id
            })

    return all_data

In [273]:
train_data = build_mtl_dataset(MMLU_TASKS, split="dev")

print(len(train_data))
print(train_data[0])

econometrics
high_school_macroeconomics
high_school_microeconomics
business_ethics
management
marketing
abstract_algebra
college_mathematics
elementary_mathematics
high_school_mathematics
high_school_statistics
55
{'instruction': 'The following is a multiple choice question. Understand the question and  select only one choice . \n\n            Question: For a stationary autoregressive process, shocks will\n Choices:\n            A. Eventually die away\n            B. Persist indefinitely\n            C. Grow exponentially\n            D. Never occur\n\n            Answer Format: A/B/C/D\n            ', 'input': '', 'output': 'A', 'task_id': 0}


In [275]:
random.shuffle(train_data)
# Save
with open("mmlu_code_train_dataset.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

In [277]:
from datasets import load_dataset

data_path = r"C:\Users\thara\mmlu_code_train_dataset.json"

data_test = load_dataset("json", data_files=data_path)
data_test

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'task_id'],
        num_rows: 55
    })
})

In [ ]:
###test mmlu ### dummy code####

In [63]:
from datasets import load_dataset

dataset = load_dataset("cais/mmlu",'abstract_algebra')  # or load local dataset

print(dataset['dev'])

Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 5
})


In [67]:
print(dataset['dev']['answer'])

[1, 1, 2, 0, 0]


In [ ]:
train_data = build_mtl_dataset(MMLU_TASKS, split="train")

print(len(train_data))
print(train_data[0])


In [ ]:
############# bigbench ##############

In [155]:
# Replace with your JSON file path
dataset_all_sentences_en = load_dataset("json", data_files=r"C:\Users\thara\Downloads\all_sentences_en.json", field="examples")



In [141]:
print(len(dataset['train']))
print(dataset['train'][1])

57466
{'input': 'This lady and our 2-month-old daughter were brought in with a suspicion of nervous system immaturity.', 'target_scores': {'ambivalent': 0, 'negative': 0, 'neutral': 1, 'positive': 0}}


In [229]:
TASK_ID = {
    "all_sentences_en": 0,
    "all_text_en": 1,
    "causal_judgment": 2,
    "formal_fallacies": 3,
    "logical_fallacy": 4,
    "stratergyqa":5,
    "navigate":6,
    "information_essentiality":7,
    "figure_of_speech":8
}

print(TASK_ID)


#TASK_ID = {task["name"]: i for i, task in enumerate(BIG_BENCH_TASKS)}

print(TASK_ID)


{'all_sentences_en': 0, 'all_text_en': 1, 'causal_judgment': 2, 'formal_fallacies': 3, 'logical_fallacy': 4, 'stratergyqa': 5, 'navigate': 6, 'information_essentiality': 7, 'figure_of_speech': 8}
{'all_sentences_en': 0, 'all_text_en': 1, 'causal_judgment': 2, 'formal_fallacies': 3, 'logical_fallacy': 4, 'stratergyqa': 5, 'navigate': 6, 'information_essentiality': 7, 'figure_of_speech': 8}


In [231]:
def build_instruction(task, sample):

    if task in ["all_sentences_en", "all_text_en"]:
        return f"Determine whether given opinion is positive, negative, neutral or ambivalent in terms of its sentiment.\n\nQuestion: {sample['input']}\n\nAnswer Format: positive/negative/neutral/ambivalent"

    if task == "causal_judgment":
        return f"How would a typical person answer each of the following questions about causation?.\n\nQuestion: {sample['input']}\n\nAnswer Format: Yes/No"

    if task == "formal_fallacies":
        return f"Distinguish deductively valid arguments from formal fallacies.\n\nQuestion: {sample['input']}\n\nAnswer Format:  Valid/Invalid"
        
    if task == "logical_fallacy":
         return f"Detect informal and formal logical fallacies.\n\nQuestion: {sample['input']}\n\nAnswer Format: Valid/Invalid"
    if task == "stratergyqa":
         return f"Answer questions in which the required reasoning steps are implicit in the question.\n\nQuestion: {sample['input']}\n\nAnswer Format: Yes/No"

    if task == "navigate":
         return f"If you follow these instructions, do you return to the starting point?.\n Instruction type: {sample['inst_type']} \nQuestion: {sample['input']}\n\nAnswer Format: True/False"
    if task == "information_essentiality":
         return f"Identify statements that are essential to answer a question.\n\nQuestion: {sample['input']}Answer Choices: A. Statement 1 alone is sufficient while statement 2 alone is insufficient B. Statement 2 alone is sufficient while statement 1 alone is insufficient C. Either statement 1 or statement 2 is sufficient D. Statement 1 and statement 2 taken together are sufficient E. Neither statement 1 nor statement 2 nor statements 1 and 2 taken together is sufficient F.The question can be answered without either statement \nAnswer Format: A/B/C/D/E/F."
        
    if task== "figure_of_speech":
        return f"Please identify the figure of speech embodied by the following English sentences. \n\nQuestion:  {sample['input']}\n\nAnswer Format: Simile/Metaphor/Personification/Apostrophe/Oxymoron/Hyperbole/Pun/Euphemism/Alliteration/Onomatopoeia"
       


In [233]:
def get_label_from_target_scores(target_scores, name):
    if name == 'information_essentiality':
        for label, val in target_scores.items():
            if val == 1:
                mapping = {
                    'Statement 1 alone is sufficient while statement 2 alone is insufficient': 'A',
                    'Statement 2 alone is sufficient while statement 1 alone is insufficient': 'B',
                    'Either statement 1 or statement 2 is sufficient': 'C',
                    'Statement 1 and statement 2 taken together are sufficient': 'D',
                    'Neither statement 1 nor statement 2 nor statements 1 and 2 taken together is sufficient': 'E',
                    'The question can be answered without either statement': 'F'
                }
                if label in mapping:
                    return mapping[label]
                else:
                    raise ValueError(f"Unknown label: {label}")
        raise ValueError("No label with value 1 found for information_essentiality")
    else:
        # fallback for other tasks
        for label, val in target_scores.items():
            if val == 1:
                return label
        raise ValueError("No label with value 1 found for task {name}")

In [235]:
from datasets import load_dataset

# Load datasets
dataset_all_sentences_en = load_dataset(
    "json",
    data_files=r"C:\Users\thara\Downloads\all_sentences_en.json",
    field="examples"
)["train"]

dataset_all_text_en = load_dataset(
    "json",
    data_files=r"C:\Users\thara\Downloads\all_text_en.json",
    field="examples"
)["train"]

dataset_causal_judgment = load_dataset(
    "json",
    data_files=r"C:\Users\thara\Downloads\causal_judgment.json",
    field="examples"
)["train"]

dataset_formal_fallacies = load_dataset(
    "json",
    data_files=r"C:\Users\thara\Downloads\formal_fallacies.json",
    field="examples"
)["train"]

dataset_logical_fallacy = load_dataset(
    "json",
    data_files=r"C:\Users\thara\Downloads\logical_fallacy.json",
    field="examples"
)["train"]

dataset_information_essentiality = load_dataset(
    "json",
    data_files=r"C:\Users\thara\Downloads\evaluating_information_essentiality.json",
    field="examples"
)["train"]

dataset_stratergyqa= load_dataset(
    "json",
    data_files=r"C:\Users\thara\Downloads\stratergyqa.json",
    field="examples"
)["train"]

dataset_navigate = load_dataset(
    "json",
    data_files=r"C:\Users\thara\Downloads\navigate.json",
    field="examples"
)["train"]

dataset_figure_of_speech = load_dataset(
    "json",
    data_files=r"C:\Users\thara\Downloads\figure_of_speech.json",
    field="examples"
)["train"]


all_sentences_en_split = dataset_all_sentences_en.train_test_split(test_size=0.2, seed=42)
all_text_en_split = dataset_all_text_en.train_test_split(test_size=0.2, seed=42)
dataset_causal_judgment_split = dataset_causal_judgment.train_test_split(test_size=0.2, seed=42)
dataset_formal_fallacies_split = dataset_formal_fallacies.train_test_split(test_size=0.2, seed=42)
dataset_logical_fallacy_split = dataset_logical_fallacy.train_test_split(test_size=0.2, seed=42)
dataset_information_essentiality_split = dataset_information_essentiality.train_test_split(test_size=0.2, seed=42)
dataset_stratergyqa_split = dataset_stratergyqa.train_test_split(test_size=0.2, seed=42)
dataset_navigate_split = dataset_navigate.train_test_split(test_size=0.2, seed=42)
dataset_figure_of_speech_split = dataset_figure_of_speech.train_test_split(test_size=0.2, seed=42)

In [237]:
# ✅ FIX 3: map task → dataset (CRITICAL FIX)
DATASET_MAP = {
    "all_sentences_en": all_sentences_en_split,
    "all_text_en": all_text_en_split,
    "causal_judgment": dataset_causal_judgment_split,
    "formal_fallacies": dataset_formal_fallacies_split,
    "logical_fallacy": dataset_logical_fallacy_split,
     "information_essentiality": dataset_information_essentiality_split,
    "stratergyqa": dataset_stratergyqa_split,
    "navigate": dataset_navigate_split,
    "figure_of_speech": dataset_figure_of_speech_split
}


In [239]:
BIG_BENCH_TASKS = [
    {"name": "all_sentences_en"},
    {"name": "all_text_en"},
    {"name": "causal_judgment"},
    {"name": "formal_fallacies"},
    {"name": "logical_fallacy"},
    {"name":  "information_essentiality"},
   {"name":  "stratergyqa"},
   {"name":  "navigate"},
    {"name": "figure_of_speech"}

]

In [245]:
def build_mtl_dataset_big(task_defs, split="train"):
    all_data = []

    for task_def in task_defs:
        name = task_def["name"]
        task_id = TASK_ID[name]

        print(f"Processing: {name}")

        ds_split = DATASET_MAP[name]

        if split == "train":
            ds = ds_split["train"]
        elif split == "validation":
            ds = ds_split["test"]
        else:
            raise ValueError("split must be 'train' or 'validation'")

        for sample in ds:
            instruction = build_instruction(name, sample)
            output = get_label_from_target_scores(sample["target_scores"],name)

            all_data.append({
                "instruction": instruction,
                "input": "",
                "output": output,   # ✅ "A"/"B"/"C"/etc
                "task_id": task_id  # ✅ correct per task
            })
        ##### this only for eval dataset
        random.shuffle(all_data)
            # Save
        file_name =  name +".json"
        with open(file_name, "w", encoding="utf-8") as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        all_data =[]
            


build_mtl_dataset_big(BIG_BENCH_TASKS, split="validation")

    #return all_data

Processing: all_sentences_en
Processing: all_text_en
Processing: causal_judgment
Processing: formal_fallacies
Processing: logical_fallacy
Processing: information_essentiality
Processing: stratergyqa
Processing: navigate
Processing: figure_of_speech


In [95]:
train_data = build_mtl_dataset_big(BIG_BENCH_TASKS, split="train")
#val_data   = build_mtl_dataset_big(BIG_BENCH_TASKS, split="validation")

Processing: all_sentences_en
Processing: all_text_en
Processing: causal_judgment
Processing: formal_fallacies
Processing: logical_fallacy
Processing: information_essentiality
Processing: stratergyqa
Processing: navigate
Processing: figure_of_speech
Processing: all_sentences_en
Processing: all_text_en
Processing: causal_judgment
Processing: formal_fallacies
Processing: logical_fallacy
Processing: information_essentiality
Processing: stratergyqa
Processing: navigate
Processing: figure_of_speech


In [97]:
import random

In [99]:
random.shuffle(train_data)
random.shuffle(val_data)
# Save
with open("bigbench_code_train_dataset.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open("bigbench_code_val_dataset.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

In [225]:
### testing bigbench
import pandas as pd

test_data = load_dataset("json", data_files=r"C:\Users\thara\bigbench_code_val_dataset.json")

DF=pd.DataFrame(test_data['train'])
# Save


Generating train split: 0 examples [00:00, ? examples/s]

In [227]:
#print(DF.head(5))
import numpy as np
#select distinct(output) from a group by task_id
pd.set_option('display.max_colwidth', None)
df_taskid =DF.groupby('task_id')

print(df_taskid['output'].unique())

#print(df_taskid.head(50))


task_id
0                                        [negative, neutral, positive, ambivalent]
1                                        [negative, positive, neutral, ambivalent]
2                                                                        [No, Yes]
3                                                                 [invalid, valid]
4                                                                 [Invalid, Valid]
5                                                                        [Yes, No]
6                                                                    [True, False]
7                                                               [E, A, D, B, F, C]
8    [Hyperbole, Alliteration, Euphemism, Personification, Onomatopoeia, Oxymoron]
Name: output, dtype: object


In [ ]:
#################### generating eval dataset ###########

In [139]:
##### glue and superglue
'''
   # {"name": "ax", "group": "glue"}, this has test split
    {"name": "mnli", "group": "glue"},
    {"name": "cola", "group": "glue"},
    {"name": "sst2", "group": "glue"},
    {"name": "mrpc", "group": "glue"},
    {"name": "qqp", "group": "glue"},
    {"name": "qnli", "group": "glue"},
    {"name": "rte", "group": "glue"},
    #{"name": "wnli", "group": "glue"}, # not in my paper
    {"name": "stsb", "group": "glue"},
    '''
GLUE_SUPERGLUE_TASKS = [
    # GLUE


        # SuperGLUE
    {"name": "multirc", "group": "superglue"},
    {"name": "wsc", "group": "superglue"},
    {"name": "boolq", "group": "superglue"},
    {"name": "cb", "group": "superglue"},
    {"name": "wic", "group": "superglue"},

    

]


TASK_ID = {task["name"]: i for i, task in enumerate(GLUE_SUPERGLUE_TASKS)}

print(TASK_ID)

{'multirc': 0, 'wsc': 1, 'boolq': 2, 'cb': 3, 'wic': 4}


In [141]:
from datasets import ClassLabel

def get_label_map(dataset):
    label_feat = dataset.features["label"]
    if isinstance(label_feat, ClassLabel):
        return {i: name for i, name in enumerate(label_feat.names)}
    return None  # regression


In [143]:
def build_instruction(task, sample):
    '''
    if task == "sst2":
        return f"Classify the sentiment of the sentence as positive or negative.\nSentence: {sample['sentence']}\n\nAnswer format: postive/negative"
    if task == "ax":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['premise']}\nSentence 2: {sample['hypothesis']}\n\nAnswer format: entailment/neutral/contradiction"
    if task == "mnli":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['premise']}\nSentence 2: {sample['hypothesis']}\n\nAnswer format: entailment/neutral/contradiction"
        
    if task == "cola":
        return f"Is the following sentence grammatically acceptable?\nSentence: {sample['sentence']}\n\nAnswer format: unacceptable/acceptable "

    if task == "mrpc":
        return f"Do the following two sentences have the same meaning?\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}\n\nAnswer format: not_equivalent/equivalent"

    if task == "qqp":
        return f"Are the following two questions duplicates?\nQuestion 1: {sample['question1']}\nQuestion 2: {sample['question2']}\n\nAnswer format: not_duplicate/duplicate"

    if task == "qnli":
        return f"Does the sentence answer the question?\nQuestion: {sample['question']}\nSentence: {sample['sentence']}\n\nAnswer format: entailment/not_entailment"

    if task == "rte":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}\n\nAnswer format: entailment/not_entailment"

    if task == "wnli":
        return f"Does the first sentence entail the second?\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}\n\nAnswer format: entailment/not_entailment"

    if task == "stsb":
        return f"Rate the semantic similarity between the two sentences on a scale from 0 to 5.\nSentence 1: {sample['sentence1']}\nSentence 2: {sample['sentence2']}"
'''
    if task == "boolq":
        return f"Answer the question based on the passage.\nPassage: {sample['passage']}\nQuestion: {sample['question']}\n\nAnswer format: False/True"
    if task=="multirc":
            return f"""Determine whether the candidate answer correctly answers the question based on the paragraph.

                    Paragraph:
                    {sample['paragraph']}
                    
                    Question: {sample['question']}
                    
                    Candidate Answer:
                    {sample['answer']}
                    
                    Answer Format: True / False
                    """
    if task == "wsc":
            return f"""Coreference Resolution Task: 

                        Sentence:
                        "{sample['text']}"
                      
                        Candidates:
                        A. {sample['span1_text']}
                        B. {sample['span2_text']}
                        Question: Does the pronoun refer to candidate A?
                         Answer: True or False. \n\nAnswer format: False/True"""

    if task =="wic":
        return f"""Decide whether the highlighted word has the same meaning in two different sentences.

                        Word: {sample['word']}
                        
                        Sentence 1: {sample['sentence1']}
                        Sentence 2: {sample['sentence2']}
                        
                        Answer Format: True/False
                        """  

    if task == "cb":
        return f"Determine the relationship between the premise and hypothesis.\nPremise: {sample['premise']}\nHypothesis: {sample['hypothesis']}\n\nAnswer format: entailment/neutral/contradiction"



    raise ValueError(f"Unknown task {task}")


In [145]:
from datasets import load_dataset, concatenate_datasets
import random
def build_mtl_dataset(task_defs, split="train"):
    all_data = []

    for task_def in task_defs:
        name = task_def["name"]
        group = task_def["group"]
        task_id = TASK_ID[name]

        if group == "glue":
            print(name)
            if name == "mnli":
                matched = load_dataset("glue", "mnli", split="validation_matched")
                mismatched = load_dataset("glue", "mnli", split="validation_mismatched")
                dataset = concatenate_datasets([matched, mismatched])
            else:
                dataset = load_dataset("glue", name, split=split)
        else:
            print(name)
            dataset = load_dataset("super_glue", name, split=split)

        label_map = get_label_map(dataset)

        for sample in dataset:
            instruction = build_instruction(name, sample)

            if label_map is None:
                output = str(float(sample["label"]))  # STS-B
            else:
                output = label_map[sample["label"]]

            all_data.append({
                "instruction": instruction,
                "input": "",
                "output": output,
                "task_id": task_id
            })
        random.shuffle(all_data)
            # Save
        file_name =  name +".json"
        with open(file_name, "w", encoding="utf-8") as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        all_data =[]
            


build_mtl_dataset(GLUE_SUPERGLUE_TASKS, split="validation")


multirc
wsc
boolq
cb
wic


In [279]:
#####  mmlu

MMLU_TASKS = [
    # MMLU
   # {"name": "ax", "group": "glue"}, this has test split
    {"name": "econometrics"},
    {"name": "high_school_macroeconomics"},
    {"name": "high_school_microeconomics"},
    {"name": "business_ethics" },
    {"name": "management"},
    {"name": "marketing"},
    {"name": "abstract_algebra"},
    {"name": "college_mathematics"}, 
    {"name": "elementary_mathematics"},
    {"name": "high_school_mathematics"}, 
    {"name": "high_school_statistics"},

    # BigBench

   # {"name": "multiemo_zero_shot", "group": "BigBench"},

    
]

#groups = set( s["group"] for s in MMLU_TASKS)

#group_to_taskid = {group: idx for idx, group in enumerate(groups)}
TASK_ID = {task["name"]: i for i, task in enumerate(MMLU_TASKS)}

print (TASK_ID)




{'econometrics': 0, 'high_school_macroeconomics': 1, 'high_school_microeconomics': 2, 'business_ethics': 3, 'management': 4, 'marketing': 5, 'abstract_algebra': 6, 'college_mathematics': 7, 'elementary_mathematics': 8, 'high_school_mathematics': 9, 'high_school_statistics': 10}


In [301]:
print( {task["name"] for  task in MMLU_TASKS})

{'elementary_mathematics', 'business_ethics', 'econometrics', 'abstract_algebra', 'college_mathematics', 'marketing', 'management', 'high_school_mathematics', 'high_school_statistics', 'high_school_macroeconomics', 'high_school_microeconomics'}


In [281]:
def build_instruction(task, sample):

    prompt=f"""The following is a multiple choice question. Understand the question and  select only one choice . 
            Question: {sample['question']}\n Choices:
            A. {sample['choices'][0]}
            B. {sample['choices'][1]}
            C. {sample['choices'][2]}
            D. {sample['choices'][3]}

            Answer Format: A/B/C/D
            """
        
    return prompt


    raise ValueError(f"Unknown task {task}")


In [283]:
from datasets import ClassLabel

def get_label_map(dataset):
    label_feat = dataset.features["answer"]
    if isinstance(label_feat, ClassLabel):
        return {i: name for i, name in enumerate(label_feat.names)}
    return None  # regression

In [285]:
from datasets import load_dataset
from collections import defaultdict

def build_mtl_dataset(task_defs, split):
    all_data = []

    for task_def in task_defs:
        name = task_def["name"]
        #group = task_def["group"]
        task_id = TASK_ID[name]

        #if group == "mmlu":
        print(name)
        dataset = load_dataset("cais/mmlu", name, split=split)
        

        label_map = get_label_map(dataset)

        for sample in dataset:
            instruction = build_instruction(name, sample)


            output = label_map[sample["answer"]]

            all_data.append({
                "instruction": instruction,
                "input": "",
                "output": output,
                "task_id": task_id
            })
        random.shuffle(all_data)
            # Save
        file_name =  name +".json"
        with open(file_name, "w", encoding="utf-8") as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        all_data =[]




build_mtl_dataset(MMLU_TASKS, split="validation")

econometrics
high_school_macroeconomics
high_school_microeconomics
business_ethics
management
marketing
abstract_algebra
college_mathematics
elementary_mathematics
high_school_mathematics
high_school_statistics


In [287]:
from datasets import load_dataset

data_path = r"C:\Users\thara\econometrics.json"

data_test = load_dataset("json", data_files=data_path)
data_test

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'task_id'],
        num_rows: 12
    })
})

In [249]:
sentence = "A. 464.05"

In [257]:
import re
sentence_ =sentence.strip()
print(" the predicted sentence is:", sentence_)
print(re.findall(r"A|B|C|D", sentence_))


 the predicted sentence is: A. 464.05
['A']


In [361]:
test= ['Below is an instruction that describes a task. Write a response that appropriately completes the request. \n\n                ### Instruction:\n                The following is a multiple choice question. Choose only one choice (A, B, C, or D). Answer only with the letter. Do not write the numer or values.\n\n            Question: Suppose that a test that the true value of the intercept coefficient is zero results in non-rejection. What would be the appropriate conclusion?\n Choices:\n            A. Drop the intercept and re-run the regression\n            B. Retain the intercept\n            C. Re-compute the test statistic\n            D. The regression line is running exactly through the origin\n\n            Answer Format: A/B/C/D\n            \n\n                ### Response:\n                \n                A\n\n                ### Instruction:\n                The following is a multiple choice question. Choose only one choice (A, B, C,']


In [363]:
[o.split("### Response:")[1].strip() for o in test]

['A\n\n                ### Instruction:\n                The following is a multiple choice question. Choose only one choice (A, B, C,']